In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import numpy as np
import librosa
from tqdm import tqdm

In [3]:
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

In [4]:
durations_jazz = []
sample_rates = set()
corrupted_files = 0
vocal_peak_db = []
spectral_centroid_genre = {}
silence_count = 0

In [5]:
for genre in os.listdir(BASE_PATH):
    
    genre_path = os.path.join(BASE_PATH, genre)
    
    if not os.path.isdir(genre_path):
        continue
    
    centroid_values = []
    
    for song_folder in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song_folder)
        
        if not os.path.isdir(song_path):
            continue
        
        for stem_file in os.listdir(song_path):
            
            file_path = os.path.join(song_path, stem_file)
            
        
            if os.path.getsize(file_path) == 0:
                corrupted_files += 1
                continue
            
            try:
                y, sr = librosa.load(file_path, sr=None)
                
        
                sample_rates.add(sr)
                
               
                if genre == "jazz":
                    duration = librosa.get_duration(y=y, sr=sr)
                    durations_jazz.append(duration)
                
                
                if "vocals" in stem_file.lower():
                    peak = np.max(np.abs(y))
                    peak_db = librosa.amplitude_to_db(np.array([peak]), ref=1.0)[0]
                    vocal_peak_db.append(peak_db)
                
               
                centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
                centroid_mean = np.mean(centroid)
                centroid_values.append(centroid_mean)
                
               
                first_half_sec = y[:int(0.5 * sr)]
                if np.max(np.abs(first_half_sec)) < 1e-4:
                    silence_count += 1
                
            except:
                corrupted_files += 1
                continue
    
    if len(centroid_values) > 0:
        spectral_centroid_genre[genre] = np.mean(centroid_values)

In [6]:
mean_duration_jazz = np.mean(durations_jazz)
print("Mean Duration (Jazz):", mean_duration_jazz, "seconds")

Mean Duration (Jazz): 30.032979591836728 seconds


In [7]:
print("Unique Sample Rates:", sorted(list(sample_rates)))

Unique Sample Rates: [44100]


In [8]:
print("Number of Corrupted Files:", corrupted_files)

Number of Corrupted Files: 0


In [9]:
mean_vocal_peak = np.mean(vocal_peak_db)
print("Average Vocal Peak Amplitude (dB):", mean_vocal_peak)

Average Vocal Peak Amplitude (dB): -12.494921


In [10]:
print("Mean Spectral Centroid (Blues):", spectral_centroid_genre.get("blues"))

Mean Spectral Centroid (Blues): 2296.7827371150856


In [11]:
highest_genre = max(spectral_centroid_genre, key=spectral_centroid_genre.get)
print("Genre with Highest Spectral Centroid:", highest_genre)

Genre with Highest Spectral Centroid: metal


In [12]:
print("Files with Silence in First 0.5 seconds:", silence_count)

Files with Silence in First 0.5 seconds: 333


In [ ]:
#Training on model(Page 2)

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report

In [2]:
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')

GENRES = ["blues", "classical", "country", "disco",
          "hiphop", "jazz", "metal", "pop",
          "reggae", "rock"]

In [12]:
def extract_features(song_path):
    file_path = os.path.join(song_path, 'other.wav')

    y, sr = librosa.load(file_path, sr=22050, duration=10)

    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))

    return [float(tempo.item()), spec_cent, zcr, rolloff]

In [9]:
data = []

for genre in GENRES:
    genre_path = os.path.join(STEMS_PATH, genre)

    songs = [s for s in os.listdir(genre_path)
             if os.path.isdir(os.path.join(genre_path, s))]

    for song in songs:
        data.append({
            "path": os.path.join(genre_path, song),
            "genre": genre
        })

df = pd.DataFrame(data)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 2)


,path,genre
0,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues
1,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues
2,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues
3,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues
4,/kaggle/input/jan-2026-dl-gen-ai-project/messy...,blues


In [10]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["genre"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 800
Validation size: 200


In [11]:
X_train = np.array([extract_features(p) for p in train_df["path"]])
y_train = train_df["genre"]

X_val = np.array([extract_features(p) for p in val_df["path"]])
y_val = val_df["genre"]

print("Feature shape:", X_train.shape)

/tmp/ipykernel_55/3997070433.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return [float(tempo), spec_cent, zcr, rolloff]


Feature shape: (800, 4)


In [13]:
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [16]:
# --- Evaluation ---

# Predicted values
y_pred = clf.predict(X_val)

# Macro F1 score
macro_f1 = f1_score(y_val, y_pred, average="macro")

# Confusion matrix
cm = confusion_matrix(y_val, y_pred, labels=GENRES)

# Classification report
cr = classification_report(y_val, y_pred)

# Print results
print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

Validation Macro F1 Score: 0.2012

Detailed Classification Report:
              precision    recall  f1-score   support

       blues       0.00      0.00      0.00        20
   classical       0.00      0.00      0.00        20
     country       0.18      0.35      0.24        20
       disco       0.26      0.25      0.26        20
      hiphop       0.60      0.15      0.24        20
        jazz       0.19      0.20      0.20        20
       metal       0.32      0.60      0.41        20
         pop       0.19      0.30      0.23        20
      reggae       0.44      0.35      0.39        20
        rock       0.05      0.05      0.05        20

    accuracy                           0.23       200
   macro avg       0.22      0.23      0.20       200
weighted avg       0.22      0.23      0.20       200



In [17]:
# Convert classification report to dictionary
report_dict = classification_report(y_val, y_pred, output_dict=True)

# Precision of hiphop
hiphop_precision = report_dict["hiphop"]["precision"]

# Recall of pop
pop_recall = report_dict["pop"]["recall"]

print("Precision of hiphop:", hiphop_precision)
print("Recall of pop:", pop_recall)

Precision of hiphop: 0.6
Recall of pop: 0.3


In [18]:
# Accuracy
accuracy = report_dict["accuracy"]

print("Model Accuracy:", accuracy)

Model Accuracy: 0.225


In [19]:
# True Positives are diagonal of confusion matrix
tp_values = np.diag(cm)

tp_dict = dict(zip(clf.classes_, tp_values))

highest_tp_genre = max(tp_dict, key=tp_dict.get)

print("True Positives per Genre:", tp_dict)
print("Genre with Highest True Positives:", highest_tp_genre)

True Positives per Genre: {'blues': np.int64(0), 'classical': np.int64(0), 'country': np.int64(7), 'disco': np.int64(5), 'hiphop': np.int64(3), 'jazz': np.int64(4), 'metal': np.int64(12), 'pop': np.int64(6), 'reggae': np.int64(7), 'rock': np.int64(1)}
Genre with Highest True Positives: metal


In [20]:
fn_values = cm.sum(axis=1) - np.diag(cm)

fn_dict = dict(zip(clf.classes_, fn_values))

lowest_fn_genre = min(fn_dict, key=fn_dict.get)

print("False Negatives per Genre:", fn_dict)
print("Genre with Lowest False Negatives:", lowest_fn_genre)

False Negatives per Genre: {'blues': np.int64(20), 'classical': np.int64(20), 'country': np.int64(13), 'disco': np.int64(15), 'hiphop': np.int64(17), 'jazz': np.int64(16), 'metal': np.int64(8), 'pop': np.int64(14), 'reggae': np.int64(13), 'rock': np.int64(19)}
Genre with Lowest False Negatives: metal
